In [1]:
#Importar vendas.csv
from google.colab import files

uploaded = files.upload()

Saving vendas.csv to vendas.csv


In [7]:
#RF02 – Inspecionar e Descrever os Dados
import pandas as pd

df = pd.read_csv("vendas.csv")

def data_inspect(df: pd.DataFrame):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(
        f"Shape: {df.shape}, sendo {df.shape[0]} linhas e {df.shape[1]} colunas"
    )
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

data_inspect(df)



=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8), sendo 200 linhas e 8 colunas

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        10
preco_unitario     4
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-11-0

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14
...,...,...,...,...,...,...,...,...
195,196,2025-06-04,CLIENTE-038,Monitor,Computadores,Centro-Oeste,5.0,NaN
196,197,2025-09-15,Cliente_046,Tablet,Celulares,Sul,3.0,1748.92
197,198,2025-08-21,Cliente_048,Smartphone,Celulares,Sul,3.0,2185.60
198,199,2025-11-02,Cliente_006,Headset,Perifericos,Norte,4.0,368.94


In [11]:
#RF03 – Limpar e Tratar os Dados
#re = lib de expressoes regulares
import re

#limpa os dados de uma copia da DF (para manter a consistencia da DF original)
def data_clean(df: pd.DataFrame):

    initial_data_count = len(df)

    for column in ["cliente", "produto", "categoria", "regiao"]:
        #remove espacos inicio/fim da string
        df[column] = df[column].str.strip()

    #converte para data alterando data invalida para NaT
    df["data_venda"] = pd.to_datetime(
        df["data_venda"],
        errors="coerce"
    )

    invalid_dates_count: int = df["data_venda"].isna().sum()

    #remove datas invalidas
    df = df.dropna(subset=["data_venda"])

    #contagem de nulos em quantidade e preco_unitario para relatorio
    critical_null_count = df[
        ["quantidade", "preco_unitario"]
    ].isnull().any(axis=1).sum()

    #descartar nulos em quantidade e preco_unitario
    df = df.dropna(
        subset=["quantidade", "preco_unitario"]
    )

    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    #limpeza de texto com expressao regular
    df["cliente"] = df["cliente"].apply(
        lambda s: re.sub(r"[^A-Za-z0-9_]", "", str(s).strip())
    )

    # Validar padrao Cliente_NNN - IGNORECASE = ignorando maiusc/munisc
    valid_client = re.compile(
        r"^Cliente_\d{3}$",
        flags=re.IGNORECASE
    )

    #informa se o cliente esta fora do padrao com true ou false
    df["cliente_fora_padrao"] = ~df["cliente"].apply(
        lambda x: bool(valid_client.match(x))
    )

    final_data_count = len(df)

    report = {
        "initial_data_count": initial_data_count,
        "invalid_dates_count": int(invalid_dates_count),
        "critical_null_count": int(critical_null_count),
        "final_data_count": final_data_count
    }

    print("\n=== RELATORIO DE LIMPEZA ===")
    print(f"Registros iniciais: {initial_data_count}")
    print(f"Datas inválidas removidas: {invalid_dates_count}")
    print(f"Nulos críticos removidos: {critical_null_count}")
    print(f"Registros finais: {final_data_count}")
    print(f"Registros removidos: {(initial_data_count - final_data_count)}")

    return df, report

data_clean(df)


=== RELATORIO DE LIMPEZA ===
Registros iniciais: 200
Datas inválidas removidas: 4
Nulos críticos removidos: 13
Registros finais: 183
Registros removidos: 17


(     id_venda data_venda      cliente     produto     categoria        regiao  \
 0           1 2025-05-21   cliente016       Mouse   Perifericos       Sudeste   
 2           3 2025-03-23  Cliente_045      Tablet     Celulares  Centro-Oeste   
 3           4 2025-11-06  Cliente_017    Notebook  Computadores         Norte   
 4           5 2025-07-05  Cliente_037      Tablet     Celulares           Sul   
 5           6 2025-08-21  Cliente_041     Headset   Perifericos       Sudeste   
 ..        ...        ...          ...         ...           ...           ...   
 194       195 2025-12-09  Cliente_007     Teclado   Perifericos           Sul   
 196       197 2025-09-15  Cliente_046      Tablet     Celulares           Sul   
 197       198 2025-08-21  Cliente_048  Smartphone     Celulares           Sul   
 198       199 2025-11-02  Cliente_006     Headset   Perifericos         Norte   
 199       200 2025-03-18  Cliente_011     Teclado   Perifericos       Sudeste   
 
      quantida